# 03. Unsupervised Clustering
This notebook clusters World Cup teams into style-of-play groups using their historical match statistics.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

df_team = pd.read_csv('../data/team_historical_features_updated.csv')
CLUSTER_FEATURES = [
    'wc_win_rate', 'avg_goals_scored', 'avg_goals_conceded',
    'goal_diff_per_match', 'wc_appearances_94_18', 'wc_titles'
]

df_cluster = df_team[df_team['wc_matches_played'] >= 3].copy()
X = df_cluster[CLUSTER_FEATURES].fillna(0)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca        = PCA(n_components=2, random_state=42)
X_pca      = pca.fit_transform(X_scaled)
df_cluster['pca1'] = X_pca[:, 0]
df_cluster['pca2'] = X_pca[:, 1]

inertias   = []
sil_scores = []
K_range    = range(2, 9)
for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))
    print(f"k={k}: Silhouette Score = {sil_scores[-1]:.3f}")

best_k = K_range[sil_scores.index(max(sil_scores))]
print(f"Best k: {best_k}")

k=2: Silhouette Score = 0.313
k=3: Silhouette Score = 0.335
k=4: Silhouette Score = 0.323
k=5: Silhouette Score = 0.282
k=6: Silhouette Score = 0.292
k=7: Silhouette Score = 0.294
k=8: Silhouette Score = 0.304
Best k: 3


### Elbow Plot and Silhouette Scores

In [2]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(K_range, inertias, marker='o', color='b')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')

plt.subplot(1, 2, 2)
plt.plot(K_range, sil_scores, marker='o', color='r')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score')

plt.tight_layout()
plt.savefig('../elbow_plot.png', dpi=150, bbox_inches='tight')
plt.show()

### Perform Final Clustering

In [3]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(X_scaled)
centroid_means = df_cluster.groupby('cluster')[CLUSTER_FEATURES].mean()
print(centroid_means)

         wc_win_rate  avg_goals_scored  ...  wc_appearances_94_18  wc_titles
cluster                                 ...                                 
0           0.562000          1.618111  ...              6.111111        2.0
1           0.068312          0.511813  ...              2.000000        0.0
2           0.301512          1.116878  ...              3.146341        0.0

[3 rows x 6 columns]


### Assign Cluster Names

In [4]:
win_rates = centroid_means['wc_win_rate'].sort_values(ascending=False)
sorted_clusters = win_rates.index.tolist()

cluster_names = {}
for rank, cluster_id in enumerate(sorted_clusters):
    if rank == 0:
        cluster_names[cluster_id] = 'Elite Powers'
    elif rank == 1:
        cluster_names[cluster_id] = 'Solid Contenders'
    elif rank == 2:
        g_scored = centroid_means.loc[cluster_id, 'avg_goals_scored']
        g_conceded = centroid_means.loc[cluster_id, 'avg_goals_conceded']
        if g_scored > g_conceded:
            cluster_names[cluster_id] = 'Attack-Heavy'
        else:
            cluster_names[cluster_id] = 'Defensive & Tight'
    else:
        cluster_names[cluster_id] = 'Developing/Inconsistent'

df_cluster['cluster_name'] = df_cluster['cluster'].map(cluster_names)
print(df_cluster[['team', 'cluster', 'cluster_name']].head(10))

df_cluster.to_csv('../data/team_clusters.csv', index=False)
print("Saved clustered teams to data/team_clusters.csv")

                  team  cluster      cluster_name
0  Republic of Ireland        2  Solid Contenders
1                Chile        2  Solid Contenders
2              Belgium        2  Solid Contenders
3             Paraguay        2  Solid Contenders
4             Bulgaria        2  Solid Contenders
5              Germany        0      Elite Powers
6               Angola        2  Solid Contenders
7               Russia        2  Solid Contenders
8                Ghana        2  Solid Contenders
9              Ukraine        2  Solid Contenders
Saved clustered teams to data/team_clusters.csv
